# [LAB-04] 12. 로그 변환

## #01. 준비작업

### 1. 라이브러리 참조

In [1]:
from jussam import load_data
import numpy as np

📦 연세대학교 주영아 교수가 제작한 라이브러리를 사용중입니다.
📧 Email(1): j.purplerose@yonsei.ac.kr
📧 Email(2): j.purplerose@gmail.com
📝 Website: https://juyounga.kr/


### 2. 데이터 불러오기

In [2]:
origin = load_data("daily_restaurant_results")
origin.head()

📚 식당의 하루 운영 결과를 기록한 데이터

    field                 description
--  --------------------  -----------------
 0  daily_customers       하루 방문 고객 수
 1  daily_sales           하루 매출액
 2  waiting_satisfaction  대기 만족도 점수



,daily_customers,daily_sales,waiting_satisfaction
0,130,7570000,92
1,117,5690000,89
2,133,3390000,93
3,150,2220000,90
4,115,4970000,97


## #02. 데이터 분포 유형 확인

### 1. 하루 방문 고객 수의 왜도와 첨도 확인

In [3]:
skew_customers = origin["daily_customers"].skew()
kurt_customers = origin["daily_customers"].kurt()
print(f"왜도: {skew_customers}, 첨도: {kurt_customers}")

왜도: 0.11963008286875651, 첨도: 0.07109561342670556


### 2. 로그 변환 필요성 판정 함수 정의

| 구분               | 조건                      | 판단                         | 권장 변환                  |
| ---------------- | ----------------------- | -------------------------- | ---------------------- |
| **강한 우측 꼬리 분포** | 왜도 >= 1                 | 양의 왜도가 뚜렷하므로 로그 변환 필요      | `log1p(x)`             |
| **우측 꼬리 분포**     | 왜도 > 0.5 and 첨도 > 0     | 약한 양의 왜도 + 극단값 영향 가능성      | `log1p(x)` 검토          |
| **강한 좌측 꼬리 분포** | 왜도 <= -1                | 음의 왜도가 뚜렷하므로 반전 후 로그 변환 필요 | `log1p(max(x) - x)`    |
| **좌측 꼬리 분포**     | 왜도 < -0.5 and 첨도 > 0    | 약한 음의 왜도 + 극단값 영향 가능성      | `log1p(max(x) - x)` 검토 |
| **대칭 또는 약한 비대칭** | 그 외                     | 로그 변환 필요 낮음                | 원자료 유지 검토              |


In [4]:
def judge_log_transform(skew, kurt):
    if skew >= 1:                   # 강한 우측 꼬리 분포
        return "log1p"
    elif skew > 0.5 and kurt > 0:   # 우측 꼬리 분포이면서 첨도가 높은 경우
        return "log1p"
    elif skew <= -1:                # 강한 좌측 꼬리 분포
        return "reverse_log1p"
    elif skew < -0.5 and kurt > 0:  # 좌측 꼬리 분포이면서 첨도가 높은 경우
        return "reverse_log1p"
    else:                           # 대칭 분포
        return "none"

In [5]:
# 함수를 호출하여 결과 확인
judge_log_transform(skew_customers, kurt_customers)

'none'

### 2. 하루 매출액에 대한 왜도와 첨도 확인

In [6]:
skew_sales = origin["daily_sales"].skew()
kurt_sales = origin["daily_sales"].kurt()
print(f"왜도: {skew_sales}, 첨도: {kurt_sales}")
judge_log_transform(skew_sales, kurt_sales)

왜도: 1.9729638697243348, 첨도: 6.372291799126957


'log1p'

### 3. 대기 만족도 점수에 대한 왜도와 첨도 확인

In [7]:
skew_waiting = origin["waiting_satisfaction"].skew()
kurt_waiting = origin["waiting_satisfaction"].kurt()
print(f"왜도: {skew_waiting}, 첨도: {kurt_waiting}")
judge_log_transform(skew_waiting, kurt_waiting)

왜도: -2.6723648970917435, 첨도: 11.50297824200852


'reverse_log1p'

## #03. 로그 변환

### 1. 하루 매출액 - 우측 꼬리 형태

In [8]:
df1 = origin.copy()
df1["log_daily_sales"] = np.log1p(df1["daily_sales"])
df1.head()

,daily_customers,daily_sales,waiting_satisfaction,log_daily_sales
0,130,7570000,92,15.840
1,117,5690000,89,15.554
2,133,3390000,93,15.036
3,150,2220000,90,14.613
4,115,4970000,97,15.419


### 2. 대기 만족도 점수 - 좌측 꼬리 형태

In [9]:
x_max = df1["waiting_satisfaction"].max()
print(f"waiting_satisfaction의 최대값: {x_max}")

df1["rev_log_waiting"] = np.log1p(x_max - df1["waiting_satisfaction"])
df1.head()

waiting_satisfaction의 최대값: 99


,daily_customers,daily_sales,waiting_satisfaction,log_daily_sales,rev_log_waiting
0,130,7570000,92,15.840,2.079
1,117,5690000,89,15.554,2.398
2,133,3390000,93,15.036,1.946
3,150,2220000,90,14.613,2.303
4,115,4970000,97,15.419,1.099


## #04. 로그 역변환

### 1. 하루 매출액 - 우측 꼬리 형태

In [10]:
df2 = df1.copy()
df2["exp_daily_sales"] = np.expm1(df2["log_daily_sales"])
df2[['daily_sales', 'log_daily_sales', 'exp_daily_sales']].head()

,daily_sales,log_daily_sales,exp_daily_sales
0,7570000,15.840,7570000.000
1,5690000,15.554,5690000.000
2,3390000,15.036,3390000.000
3,2220000,14.613,2220000.000
4,4970000,15.419,4970000.000


### 2. 대리 만족도 점수 - 좌측 꼬리 형태

In [11]:
df2['exp_rev_log_waiting'] = x_max - np.expm1(df2["rev_log_waiting"])
df2[['waiting_satisfaction', 'rev_log_waiting', 'exp_rev_log_waiting']].head()

,waiting_satisfaction,rev_log_waiting,exp_rev_log_waiting
0,92,2.079,92.000
1,89,2.398,89.000
2,93,1.946,93.000
3,90,2.303,90.000
4,97,1.099,97.000
